# NPC AI Framework — Colab Training

Runs `training/train_adapter.py` on a Colab GPU instead of the local MX450 (thermal-throttle issues, see `Docs/TODO.md` Phase 3 for the full incident log).

**Status so far:** r8/r16/r32 + r8_e8 all trained successfully on Colab. All showed zero PDM-measured archaic dialect markers (thee/thou/hath/etc) even after fixing a real bug (completion-only loss wasn't enabled — see TODO.md). Qualitatively the outputs *are* medieval-themed and coherent, just not lexically archaic. Working theory: the 1003-entry training mix dilutes the real archaic-pronoun signal (only the 626 Gutenberg-sourced entries are 100% dialect-dense; chimbiwide + hand-authored entries are medieval-themed but often pronoun-light).

**This run tests that theory** — train on ONLY the 626 Gutenberg entries via `--id-prefix GUT-`.

**Before running:** Runtime > Change runtime type > GPU (T4 is fine, free tier).

**What this notebook does:** clone the repo, install deps, log into W&B, run training, save adapters to Google Drive (Colab's local disk is wiped when the runtime recycles).

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
!git clone https://github.com/spiicez21/Final-Year-Project.git repo
%cd repo

## 2. Install dependencies

Colab ships an old `transformers`/`peft`/`trl` — pin to versions matching what was validated locally so behavior (chat template, SFTConfig API, etc.) matches.

In [ ]:
!pip install -q -U transformers==5.12.1 peft==0.19.1 trl==1.7.0 accelerate==1.14.0 datasets==5.0.0 bitsandbytes==0.49.2 wandb==0.28.0 pyyaml

## 3. Mount Drive (adapters get saved here so they survive runtime resets)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ADAPTERS_DIR = '/content/drive/MyDrive/npc-ai-framework/adapters'
os.makedirs(DRIVE_ADAPTERS_DIR, exist_ok=True)

## 4. W&B login

Paste your API key from wandb.ai/authorize when prompted.

In [ ]:
import wandb
wandb.login()

## 5. Previously completed runs (reference only — already done)

r8_e8 and r32 both completed here already (before and after the completion-only-loss fix). Skip to section 6 for the new experiment.

In [ ]:
# Run 1: 8 epochs at r=8 — the one that couldn't finish locally
!python training/train_adapter.py --domain medieval --lora-r 8 --epochs 8

In [ ]:
# Copy the finished adapter to Drive before the runtime can recycle it
!cp -r training/adapters/medieval_r8_e8 "{DRIVE_ADAPTERS_DIR}/"

In [ ]:
# Sanity check — same script used locally, same DIALECT_PATTERNS scorer
!python training/quick_inference.py --adapter training/adapters/medieval_r8_e8

In [ ]:
# Run 2: r=32 — completes the planned rank ablation (r=4/8/16/32)
!python training/train_adapter.py --domain medieval --lora-r 32
!cp -r training/adapters/medieval_r32 "{DRIVE_ADAPTERS_DIR}/"
!python training/quick_inference.py --adapter training/adapters/medieval_r32

In [ ]:
# If reusing an existing Colab session/repo clone, pull the latest fixes
# (completion-only-loss fix + --id-prefix support) before continuing.
!git pull

## 6. Dialect-density experiment — train on Gutenberg-only subset

Tests whether concentrating training on only the genuinely archaic-pronoun-dense entries (626 GUT-* ids, 100% have real dialect markers, vs 79.5% across the full 1003-entry mix) produces PDM-measurable markers where the full-mix runs didn't.

In [ ]:
!python training/train_adapter.py --domain medieval --id-prefix GUT-
!cp -r training/adapters/medieval_r8_gutonly "{DRIVE_ADAPTERS_DIR}/"
!python training/quick_inference.py --adapter training/adapters/medieval_r8_gutonly

## 7. Condition D — full fine-tune (no LoRA)

Completes the comparison matrix (A: no adapter, B: LoRA, D: full fine-tune) without needing an API key — only Condition C (GPT-4o few-shot) is still blocked on that.

Uses `--id-prefix GUT-` to match `medieval_r8_gutonly`, the config that actually produced archaic markers — otherwise Condition B vs D wouldn't be comparable. Needs more VRAM than the LoRA runs (full 1.1B params + optimizer state, not just ~1M LoRA params) — smaller per-device batch (2 vs 4) and 8-bit Adam to fit on a T4. Cannot run on the local MX450 at all (2.15GB VRAM, full fine-tune needs ~8-12GB minimum even with 8-bit Adam + gradient checkpointing).

In [ ]:
!git pull  # get train_full_finetune.py if reusing an existing clone
!python training/train_full_finetune.py --domain medieval --id-prefix GUT-
!cp -r training/adapters/medieval_full_finetune_gutonly "{DRIVE_ADAPTERS_DIR}/"
!python training/quick_inference.py --adapter training/adapters/medieval_full_finetune_gutonly

## 8. Full held-out evaluation for Condition D

Same rigorous 171-entry held-out comparison Condition B got — run this right after training while the model is still in this runtime (avoids re-downloading 11.8GB from Drive). `evaluation/results/baseline_heldout_outputs.json` is committed to the repo, so `git pull` (already done above) has it.

In [ ]:
!python evaluation/run_condition_b.py --adapter training/adapters/medieval_full_finetune_gutonly --baseline-name baseline_heldout --output-name condition_d_heldout